# TestGiuse Phasor + Fit Workflow


### This Notebook's Operations

This workflow starts from `FILE_DATA`, creates a calibrated source file, then writes phasor and fit products into one copied pipeline file.

1. `FILE_DATA` -> calibration -> `FILE_WITH_CALIBRATION`
2. `FILE_WITH_CALIBRATION` -> `phasor` with `mode="copy"` -> `EXPORT_PATH`
3. `EXPORT_PATH` -> `lifetime_fit` with `mode="append"` -> same `EXPORT_PATH`

The exported file keeps the calibrated input content and adds both `/output/phasor` and `/output/lifetime_fit` products.

Clean standalone workflow for loading a calibrated HDF5 file, computing corrected phasor maps, preparing aligned summed decays, generating per-pixel fit maps, and visualizing the lifetime result.


In [ ]:
FILE_REFERENCE = '/mnt/DATA/Mixed Data/TestGiuse/FLIMLabs_DFD.h5'
FILE_DATA = '/mnt/DATA/Mixed Data/TestGiuse/Convallaria_DFD.h5'
DATA_KEY = "data"

TAU_REF = 2.5
REFERENCE_TYPE = 'ref'
FIT_MODE = "model_shift"
FIT_TYPE = "likelihood"
LASER_FREQ_MHZ = None
LASER_PERIOD_NS = None
CHANNEL_SKEW_SOURCE = 'ref'
OVERWRITE = True

THRESHOLD = 0.05
FIT_INITIAL_TAU = 4.0
FIT_INITIAL_dT = 1.0
FIT_INITIAL_C = 1.0
FIT_MODE_PIXEL = "irf_shift"
FIT_TYPE_PIXEL = "likelihood"

EXPORT_PATH = FILE_DATA.replace(".h5", "_fit_workflow.h5")
FILE_WITH_CALIBRATION = FILE_DATA.replace(".h5", "_calib.h5")

print(f"Input file (ref): {FILE_REFERENCE}")
print(f"Input file (data): {FILE_DATA}")
print(f"Calibration results to: {FILE_WITH_CALIBRATION}")
print(f"Output products results to: {EXPORT_PATH}")


In [ ]:
from pathlib import Path
import importlib
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = str(ROOT / "src")
if SRC in sys.path:
    sys.path.remove(SRC)
sys.path.insert(0, SRC)
MCS_FILE_SRC = ROOT.parent / "BrightEyes-MCS-File" / "src"
if MCS_FILE_SRC.exists():
    MCS_FILE_SRC = str(MCS_FILE_SRC)
    if MCS_FILE_SRC in sys.path:
        sys.path.remove(MCS_FILE_SRC)
    sys.path.insert(0, MCS_FILE_SRC)

import h5py
import matplotlib.pyplot as plt
import numpy as np

import brighteyes_mcs_file.alignment as alignment_module
import brighteyes_flim.graph_tools as graph
import brighteyes_flim.tools_phasor as flim
alignment_module = importlib.reload(alignment_module)
graph = importlib.reload(graph)

from brighteyes_mcs_file.alignment import Alignment
from brighteyes_mcs_file import (
    calibrate_h5_file,
    H5OutputProduct,
    load_default_irf,
    load_default_output_spad,
    load_default_ref,
    show_h5_structure_html,
    write_h5_output_run,
)

EXPECTED_H5_DATA_FORMAT_VERSION = "0.0.6"


def assert_h5_schema_version(filename, expected=EXPECTED_H5_DATA_FORMAT_VERSION):
    with h5py.File(filename, "r") as hf:
        actual = hf.attrs["data_format_version"]
    actual = actual.decode() if isinstance(actual, bytes) else str(actual)
    if actual != expected:
        raise ValueError(f"Expected H5 schema version {expected}, found {actual} in {filename}")
    return actual


In [ ]:
# Derived HDF5 schema keys. Do not edit unless the schema changes.
DATA_KEY_TO_CALIBRATION_PRODUCT = {"data": "spad", "data_channels_extra": "aux"}
INSPECT_DATA_KEY = DATA_KEY if isinstance(DATA_KEY, str) else DATA_KEY[0]
CALIBRATION_PRODUCT = DATA_KEY_TO_CALIBRATION_PRODUCT[INSPECT_DATA_KEY]


## Calibration


In [ ]:
calibrate_h5_file(
    FILE_DATA,
    FILE_REFERENCE,
    data_key=DATA_KEY,
    reference_type=REFERENCE_TYPE,
    tau_ref=TAU_REF,
    fit_mode=FIT_MODE,
    fit_type=FIT_TYPE,
    clean_irf=False,
    channel_skew_type="phase_cross_correlation",
    channel_skew_source=CHANNEL_SKEW_SOURCE,
    channel_skew_fit_reference_channel=12,
    channel_skew_fit_upsampling=10,
    channel_skew_fit_apodize=False,
    period_ns=LASER_PERIOD_NS,
    output_path=FILE_WITH_CALIBRATION,
    overwrite=OVERWRITE,
)
h5_schema_version = assert_h5_schema_version(FILE_WITH_CALIBRATION)


## Inspect Calibration File


In [ ]:
#show_h5_structure_html(FILE_WITH_CALIBRATION)


## Load Data And Metadata


In [ ]:
with h5py.File(FILE_WITH_CALIBRATION, "r") as hf:
    calibration = hf["calibration/results/spad"]
    metadata = hf["raw/metadata"].attrs

    laser_frequency_mhz = float(calibration.attrs["laser_frequency_mhz"])
    laser_period_ns = float(calibration.attrs["laser_period_ns"])
    nbin = int(metadata["time_bins"])
    pixel_size_x_um = float(metadata["pixel_size_x_um"])
    pxdwelltime = float(metadata["pixel_dwell_time_us"])

    data_input = hf["raw/spad"][:]
    aligned_irf_trace = calibration["aligned/irf_trace"][:]
    aligned_reference_trace = calibration["aligned/reference_trace"][:]

    data_summed = load_default_output_spad(hf, selection=np.s_[0, 0, ...])
    irf_summed = load_default_irf(hf)
    ref_summed = load_default_ref(hf)

t = np.arange(nbin, dtype=float) * laser_period_ns / nbin

h5_dataset_phasor_pix = flim.calculate_phasor(data_summed, harmonic=1)
irf_phasor = flim.calculate_phasor(irf_summed, threshold=0, harmonic=1)
ref_phasor = flim.calculate_phasor(ref_summed, threshold=0, harmonic=1)

print(f"Using calibrated laser timing: {laser_frequency_mhz:.4f} MHz ({laser_period_ns:.4f} ns)")
print("data_input:", data_input.shape)
print("aligned_irf_trace:", aligned_irf_trace.shape)
print("aligned_reference_trace:", aligned_reference_trace.shape)
print("data_summed:", data_summed.shape)
print("irf_summed:", irf_summed.shape)
print("ref_summed:", ref_summed.shape)
print("phasor map:", h5_dataset_phasor_pix.shape)
print("pixel_size_x_um:", pixel_size_x_um)
print("pxdwelltime:", pxdwelltime)


## Correct Phasors And Build Lifetime Maps


In [ ]:
intensity_map = data_summed.sum(axis=-1)

with h5py.File(FILE_WITH_CALIBRATION, "r") as hf:
    calibration = hf["calibration/results/spad"]
    tau_reference_ns = calibration["fit/tau_reference_ns"][:]

tau_reference_ns_median = float(np.median(tau_reference_ns))

if REFERENCE_TYPE == "ref":
    # Theoretical monoexponential reference phasor.
    omega = 2 * np.pi * laser_frequency_mhz * 1e6
    tau_ref_s = tau_reference_ns_median * 1e-9
    ref_phasor_theoretical = 1.0 / (1.0 - 1j * omega * tau_ref_s)
    correction = ref_phasor / ref_phasor_theoretical

    print(f"Ref phasor measured:    {ref_phasor:.6f}  mag. {np.abs(ref_phasor):.6f} @ {np.angle(ref_phasor, deg=True):.2f} deg")
    print(f"Ref phasor theoretical: {ref_phasor_theoretical:.6f}  mag. {np.abs(ref_phasor_theoretical):.6f} @ {np.angle(ref_phasor_theoretical, deg=True):.2f} deg")
else:
    correction = irf_phasor

print(f"Correction factor:      {correction:.6f}  mag. {np.abs(correction):.6f} @ {np.angle(correction, deg=True):.2f} deg")

phasor_map = h5_dataset_phasor_pix / correction
tau_m_map = flim.calculate_tau_m(phasor_map, dfd_freq=laser_frequency_mhz * 1e6) * 1e9
tau_phi_map = flim.calculate_tau_phi(phasor_map, dfd_freq=laser_frequency_mhz * 1e6) * 1e9
lifetime_map = tau_m_map

thresholded_phasor_map = flim.threshold_phasor(intensity_map, phasor_map, THRESHOLD)
thresholded_intensity_map = flim.threshold_intensity(intensity_map, THRESHOLD)
thresholded_tau_map, _, phasor_lifetime_mask = graph.threshold_lifetime_map(
    lifetime_map,
    intensity=intensity_map,
    threshold=THRESHOLD,
)

print("IRF phasor:", irf_phasor)
print("tau reference median (ns):", tau_reference_ns_median)
print("thresholded phasor pixels:", thresholded_tau_map.size)


In [ ]:
fig, ax = flim.plot_phasor(
    thresholded_phasor_map,
    quadrant="first",
    bins_2dplot=400,
    cmap="viridis",
    dfd_freq=laser_frequency_mhz * 1e6,
)
ax.set_title("Corrected phasor distribution")


## Optional Phasor Export

Save the phasor outputs first so the later fit export can append to the same pipeline file.


In [ ]:
if EXPORT_PATH is not None:
    phasor_source_path = PIPELINE_H5 if "PIPELINE_H5" in globals() else FILE_WITH_CALIBRATION
    phasor_mode = "append" if Path(phasor_source_path) == Path(EXPORT_PATH) else "copy"
    phasor_output_path = None if phasor_mode == "append" else EXPORT_PATH
    common_attrs = {
        "source_data_path": f"/raw/{CALIBRATION_PRODUCT}",
        "source_calibration_path": f"/calibration/results/{CALIBRATION_PRODUCT}",
        "source_metadata_path": "/raw/metadata",
        "source_axes_path": "/raw/axes",
        "time_axis_source": "/raw/axes/digital_time_ns",
        "time_bin_ns": laser_period_ns / nbin,
        "laser_frequency_mhz": laser_frequency_mhz,
        "laser_period_ns": laser_period_ns,
    }
    products = [
        H5OutputProduct("intensity", intensity_map, attrs={**common_attrs, "data_role": "integrated_intensity", "axis_order": "y,x"}),
        H5OutputProduct("phasor_real", np.real(phasor_map), attrs={**common_attrs, "data_role": "phasor_real", "axis_order": "y,x"}),
        H5OutputProduct("phasor_imag", np.imag(phasor_map), attrs={**common_attrs, "data_role": "phasor_imag", "axis_order": "y,x"}),
        H5OutputProduct("tau_m_ns", tau_m_map, attrs={**common_attrs, "data_role": "tau_m", "units": "ns", "axis_order": "y,x"}),
        H5OutputProduct("tau_phi_ns", tau_phi_map, attrs={**common_attrs, "data_role": "tau_phi", "units": "ns", "axis_order": "y,x"}),
        H5OutputProduct("tau_ns", lifetime_map, attrs={**common_attrs, "data_role": "primary_lifetime", "units": "ns", "axis_order": "y,x"}),
        H5OutputProduct("decay_sum", data_summed, attrs={**common_attrs, "data_role": "channel_summed_decay", "axis_order": "y,x,time_bin"}),
        H5OutputProduct("irf", irf_summed, attrs={**common_attrs, "data_role": "channel_summed_irf", "axis_order": "time_bin"}),
        H5OutputProduct("ref", ref_summed, attrs={**common_attrs, "data_role": "channel_summed_reference", "axis_order": "time_bin"}),
        H5OutputProduct("irf_phasor_real", np.asarray(np.real(irf_phasor)), attrs={**common_attrs, "data_role": "irf_phasor_real"}),
        H5OutputProduct("irf_phasor_imag", np.asarray(np.imag(irf_phasor)), attrs={**common_attrs, "data_role": "irf_phasor_imag"}),
        H5OutputProduct("ref_phasor_real", np.asarray(np.real(ref_phasor)), attrs={**common_attrs, "data_role": "ref_phasor_real"}),
        H5OutputProduct("ref_phasor_imag", np.asarray(np.imag(ref_phasor)), attrs={**common_attrs, "data_role": "ref_phasor_imag"}),
        H5OutputProduct("corr_real", np.asarray(np.real(correction)), attrs={**common_attrs, "data_role": "phasor_correction_real"}),
        H5OutputProduct("corr_imag", np.asarray(np.imag(correction)), attrs={**common_attrs, "data_role": "phasor_correction_imag"}),
    ]
    target_path, output_run_id = write_h5_output_run(
        phasor_source_path,
        "phasor",
        products,
        mode=phasor_mode,
        output_path=phasor_output_path,
        output_key_overwrite=True,
        output_type="image_tool",
        tool_name="Phasor lifetime",
        algorithm_name="phasor_correction",
        parameters={
            "reference_type": REFERENCE_TYPE,
            "tau_ref_input_ns": TAU_REF,
            "tau_ref_used_ns": tau_reference_ns_median,
            "threshold_fraction": THRESHOLD,
            "data_key": DATA_KEY,
            "calibration_product": CALIBRATION_PRODUCT,
        },
        metadata={
            **common_attrs,
            "pixel_size_x_um": pixel_size_x_um,
            "pixel_dwell_time_us": pxdwelltime,
            "output_ny": intensity_map.shape[0],
            "output_nx": intensity_map.shape[1],
            "output_time_bins": data_summed.shape[-1],
            "selected_channel_count": data_input.shape[-1],
            "channel_aggregation": "sum_channels_with_skew_correction",
        },
        axes=[H5OutputProduct("time_ns", t, attrs={"axis": "time", "units": "ns"})],
        attrs={
            **common_attrs,
            "input_axis_order": "repetition,z,y,x,time_bin,detector_channel",
            "output_axis_order": "y,x",
            "output_data_path": "/output/{run_id}/products/tau_ns",
        },
        set_default=True,
    )
    PIPELINE_H5 = target_path
    print(f"Saved {output_run_id} -> {target_path}")
else:
    print("Phasor export skipped (EXPORT_PATH = None).")


## Sum Channels With The Calibration Shifts


In [ ]:
data_summed_no_alignment = np.sum(data_input, axis=(0, 1, 2, 3, 5))
irf_summed_no_alignment = np.sum(aligned_irf_trace, axis=-1)

print("data_summed:", data_summed.shape)
print("irf_summed:", irf_summed.shape)


In [ ]:
graph.plot_channel_skew_correction(
    irf_no_alignment=irf_summed_no_alignment,
    irf_aligned=irf_summed,
    data_no_alignment=data_summed_no_alignment,
    data_aligned=data_summed.sum(axis=(0, 1)),
)


## Check One Pixel Fit


In [ ]:
preview_y = data_summed.shape[0] // 2
preview_x = data_summed.shape[1] // 2
preview_hist = data_summed[preview_y, preview_x, :]

preview_result, preview_cov = Alignment.perform_fit_data(
    t=t,
    data=preview_hist,
    irf=irf_summed,
    period=laser_period_ns,
    initial_tau=4.0,
    initial_dT=1.0,
    initial_C=1.0,
    mode='irf_shift',
    fit_type='likelihood',
    force_C_normalized=True,
)

fit_preview = Alignment.fit_model_data(
    t,
    preview_result["C"],
    preview_result["dT"],
    preview_result["tau"],
    irf=irf_summed,
    period=laser_period_ns,
    mode='irf_shift',
)

fit_preview = Alignment.to_numpy_1d(fit_preview, dtype=float)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t, preview_hist / preview_hist.sum(), label="Data", color="black")
ax.plot(t, irf_summed / irf_summed.sum(), label="IRF", color="tab:blue")
ax.plot(t, fit_preview / fit_preview.sum(), label="Fit", color="tab:red")
ax.set_yscale("log")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Normalized counts")
ax.set_title(f"Preview pixel y={preview_y}, x={preview_x}; tau={preview_result['tau']:.2f} ns")
ax.legend()
preview_result


## Generate Fit Maps


In [ ]:
if not hasattr(Alignment, "generate_fit_maps"):
    raise RuntimeError(
        "The notebook kernel still has an old brighteyes_mcs_file.alignment module loaded. "
        "Rerun the first import cell, or restart the kernel and run from the top."
    )

fit_maps = Alignment.generate_fit_maps(
    data=data_summed,
    irf=irf_summed,
    t=t,
    period=laser_period_ns,
    initial_tau=4.0,
    initial_dT=1.0,
    initial_C=1.0,
    mode='irf_shift',
    fit_type='likelihood',
    force_C_normalized=True,
    n_jobs=-1,
)

fit_stack, fit_stack_names = Alignment.fit_maps_to_stack(fit_maps)
print("fit_stack shape:", fit_stack.shape)
print("fit_stack axis 0:", fit_stack_names)


## Visualize Lifetime Fit


In [ ]:
tau = fit_maps["tau"]
intensity = data_summed.sum(axis=-1)

thresholded_tau, thresholded_intensity, lifetime_mask = graph.threshold_lifetime_map(
    tau,
    intensity=intensity,
    threshold=0.05,
)

print("finite tau pixels:", np.count_nonzero(np.isfinite(tau)))
print("thresholded pixels:", thresholded_tau.size)


In [ ]:
graph.plot_lifetime_summary(
    intensity=intensity,
    lifetime=tau,
    pxsize=pixel_size_x_um,
    pxdwelltime=pxdwelltime,
    lifetime_bounds=[1.0, 5.0],
    crop=30,
    threshold=0.05,
    bins=500,
    colormap="turbo",
    weighted_histogram=True,
)


In [ ]:
graph.plot_equalized_lifetime_summary(
    intensity=intensity,
    lifetime=tau,
    pxsize=pixel_size_x_um,
    pxdwelltime=pxdwelltime,
    lifetime_bounds=[1.0, 8.0],
    crop=30,
    threshold=0.05,
    bins=500,
    colormap="turbo",
    equalization_reference=thresholded_tau,
    equalization_strength=4.0,
    equalization_bins=4096,
    colorbar_ticks=12,
)


## Optional Export

Save the fit outputs as a current-schema `/output/lifetime_fit` run. The first saved step copies the calibrated input file; later pipeline steps should append to `EXPORT_PATH`.


In [ ]:
if EXPORT_PATH is not None:
    fit_source_path = PIPELINE_H5 if "PIPELINE_H5" in globals() else FILE_WITH_CALIBRATION
    fit_mode = "append" if Path(fit_source_path) == Path(EXPORT_PATH) else "copy"
    fit_output_path = None if fit_mode == "append" else EXPORT_PATH
    common_attrs = {
        "source_data_path": f"/raw/{CALIBRATION_PRODUCT}",
        "source_calibration_path": f"/calibration/results/{CALIBRATION_PRODUCT}",
        "source_metadata_path": "/raw/metadata",
        "source_axes_path": "/raw/axes",
        "time_axis_source": "/raw/axes/digital_time_ns",
        "time_bin_ns": laser_period_ns / nbin,
        "laser_frequency_mhz": laser_frequency_mhz,
        "laser_period_ns": laser_period_ns,
    }
    products = [
        H5OutputProduct("intensity", intensity, attrs={**common_attrs, "data_role": "integrated_intensity", "axis_order": "y,x"}),
        H5OutputProduct("tau_ns", tau, attrs={**common_attrs, "data_role": "lifetime_tau", "units": "ns", "axis_order": "y,x"}),
        H5OutputProduct("fit_params", fit_stack, attrs={**common_attrs, "data_role": "fit_parameter_stack", "axis_order": "fit_parameter,y,x", "fit_stack_names_json": fit_stack_names}),
        H5OutputProduct("decay_sum", data_summed, attrs={**common_attrs, "data_role": "channel_summed_decay", "axis_order": "y,x,time_bin"}),
        H5OutputProduct("irf", irf_summed, attrs={**common_attrs, "data_role": "channel_summed_irf", "axis_order": "time_bin"}),
    ]
    for name, arr in fit_maps.items():
        arr = np.asarray(arr)
        if arr.dtype.kind not in "fiu":
            continue
        products.append(
            H5OutputProduct(
                f"fit_maps/{name}",
                arr,
                attrs={**common_attrs, "data_role": f"fit_map_{name}", "axis_order": "y,x"},
            )
        )
    print("Products to export:")
    for product in products:
        arr = np.asarray(product.data)
        print(f"  {product.name}: shape {arr.shape}, dtype {arr.dtype}")

    target_path, output_run_id = write_h5_output_run(
        fit_source_path,
        "lifetime_fit",
        products,
        mode=fit_mode,
        output_path=fit_output_path,
        output_key_overwrite=True,
        output_type="image_tool",
        tool_name="Lifetime fit",
        algorithm_name="Alignment.generate_fit_maps",
        parameters={
            "fit_initial_tau_ns": FIT_INITIAL_TAU,
            "fit_initial_dT_ns": FIT_INITIAL_dT,
            "fit_initial_C": FIT_INITIAL_C,
            "fit_mode_pixel": FIT_MODE_PIXEL,
            "fit_type_pixel": FIT_TYPE_PIXEL,
            "threshold_fraction": THRESHOLD,
            "fit_stack_names": fit_stack_names,
            "data_key": DATA_KEY,
            "calibration_product": CALIBRATION_PRODUCT,
        },
        metadata={
            **common_attrs,
            "pixel_size_x_um": pixel_size_x_um,
            "pixel_dwell_time_us": pxdwelltime,
            "output_ny": data_summed.shape[0],
            "output_nx": data_summed.shape[1],
            "output_time_bins": data_summed.shape[-1],
            "selected_channel_count": data_input.shape[-1],
            "channel_aggregation": "sum_channels_with_skew_correction",
        },
        axes=[H5OutputProduct("time_ns", t, attrs={"axis": "time", "units": "ns"})],
        attrs={
            **common_attrs,
            "input_axis_order": "repetition,z,y,x,time_bin,detector_channel",
            "output_axis_order": "y,x,time_bin",
            "output_data_path": "/output/{run_id}/products/tau_ns",
        },
        set_default=True,
    )
    PIPELINE_H5 = target_path
    print(f"Saved {output_run_id} -> {target_path}")
else:
    print("Export skipped (EXPORT_PATH = None).")


In [ ]:
show_h5_structure_html(EXPORT_PATH if EXPORT_PATH is not None else FILE_WITH_CALIBRATION)
